# Business Pulse ML — урок 01: когда падение выручки действительно аномально?

**Цель:** с нуля построить и проверить простое правило предупреждений. Оно будет нашим **baseline** — ML позже должен доказать, что лучше.

Данные: `data/demo/daily_sales.csv` — **синтетические**, единицы измерения условные. Данные не показывают реальные убытки, не имеют размеченных бизнес-инцидентов и не дают оснований заявлять о точности обнаружения реальных проблем.

**Запуск:** открой папку `business_pulse_ml` в VS Code / Jupyter, выбери Python-окружение из `requirements.txt`, затем выполняй ячейки по порядку. Если Jupyter запущен из папки `lessons`, в следующей ячейке путь автоматически определяется.

## 1. Прежде чем строить модель — осмотри данные

Твоя гипотеза: аномалия — редкое отклонение **от ожидаемого поведения в сопоставимых условиях**. Наш первый способ учесть сезонность — сравнение с тем же днём недели.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px

# Работает, если ноутбук открыт из корня проекта или из папки lessons/.
root = Path.cwd()
if not (root / "data" / "demo" / "daily_sales.csv").exists():
    root = root.parent
source = root / "data" / "demo" / "daily_sales.csv"
assert source.exists(), f"Не найден файл: {source}"

sales = pd.read_csv(source, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
sales["weekday"] = sales["date"].dt.dayofweek # Пн = 0, Вс = 6
print("Дней:", len(sales), "| период:", sales.date.min().date(), "—", sales.date.max().date())
print("Нулевых дней:", int((sales.revenue == 0).sum()))
display(sales.head(5))

In [ ]:
weekday_names = ["Понедельник", "Вторник", "Среда", "Четверг", "Пятница", "Суббота", "Воскресенье"]
weekday_summary = (sales.groupby("weekday")["revenue"]
                   .agg(count="count", median="median", mean="mean", std="std"))
weekday_summary.index = [weekday_names[i] for i in weekday_summary.index]
display(weekday_summary.round(1))

fig = px.box(sales, x="weekday", y="revenue", points="outliers",
             title="Разброс выручки по дням недели: почему нельзя сравнивать воскресенье с субботой")
fig.update_xaxes(tickvals=list(range(7)), ticktext=weekday_names)
fig.show()

**Сначала ответь самому себе:** день с выручкой 400 условных единиц — это проблема? Ответ зависит как минимум от дня недели, исторического разброса и качества загрузки.

## 2. Baseline: медиана четырёх предыдущих аналогичных дней

Для даты `t` сравниваем выручку с `t−7`, `t−14`, `t−21`, `t−28` днями. **Ни одной продажи из будущего в признаках нет**. Медиана вместо среднего уменьшает влияние разового всплеска.

In [ ]:
df = sales.copy()
for lag in [7, 14, 21, 28]:
    df[f"lag_{lag}"] = df["revenue"].shift(lag)

lag_columns = ["lag_7", "lag_14", "lag_21", "lag_28"]
df["expected"] = df[lag_columns].median(axis=1)
# Нужны все 4 исторические точки, поэтому первые 28 дней не оцениваем.
df = df.dropna(subset=lag_columns).reset_index(drop=True)

df["shortfall"] = (df["expected"] - df["revenue"]).clip(lower=0)
df["drop_pct"] = np.where(df["expected"] > 0,
                           100 * df["shortfall"] / df["expected"].clip(lower=1),
                           0.0)
display(df[["date", "weekday", "revenue", "expected", "shortfall", "drop_pct"]].head(7).round(1))

### Что означают числа?

- `expected` — типичная выручка по выбранному правилу, **не доказанная истинная норма**.
- `shortfall` — насколько факт ниже ожидания. Это **не бухгалтерский убыток** и не доказанная упущенная выручка.
- `drop_pct` — относительная величина отклонения.

Порог для тревоги выбирать исключительно на тестовых данных нельзя: мы подгоним правило под те дни, на которых потом заявляем его качество.

## 3. Калибровка порога только на прошлом

Первые 140 дней с рассчитанным прогнозом — период настройки. Остальное — исторический период проверки. Возьмём 90-й процентиль величины недобора выручки в периоде настройки и дополнительный минимум 25% падения. **Это эвристика**, а не «90%-ная вероятность поломки» и не 10% гарантированных ложных тревог.

In [ ]:
calibration = df.iloc[:140].copy()
evaluation = df.iloc[140:].copy()
min_drop_pct = 25.0
absolute_limit = float(calibration["shortfall"].quantile(0.90))

evaluation["alert"] = ((evaluation["shortfall"] > absolute_limit) &
                        (evaluation["drop_pct"] >= min_drop_pct))
print(f"Калибровка: {calibration.date.min().date()} — {calibration.date.max().date()}")
print(f"Проверка:   {evaluation.date.min().date()} — {evaluation.date.max().date()}")
print(f"Абсолютный порог: {absolute_limit:.2f} усл. ед.; относительный: {min_drop_pct}%")
print(f"Кандидатов на проверку: {evaluation['alert'].sum()} из {len(evaluation)} дней")
display(evaluation.loc[evaluation.alert,
    ["date", "revenue", "expected", "shortfall", "drop_pct", "orders"]].round(1).head(12))

## 4. Посмотри на результат, а не только на цифру метрики

График ниже показывает последние 70 дней проверки и точки, где сработало наше правило.

In [ ]:
view = evaluation.tail(70)
fig = px.line(view, x="date", y=["revenue", "expected"],
              title="Факт против ожидаемой выручки: последние 70 дней")
alarms = view.loc[view.alert]
fig.add_scatter(x=alarms.date, y=alarms.revenue,
                mode="markers", name="Кандидат на аномалию",
                marker={"size": 11, "symbol": "x"})
fig.show()

mae = float(np.mean(np.abs(evaluation.revenue - evaluation.expected)))
wape = float(100 * np.abs(evaluation.revenue - evaluation.expected).sum() /
             max(1.0, evaluation.revenue.abs().sum()))
print(f"MAE = {mae:.2f} усл. ед. | WAPE = {wape:.2f}%")
print("Внимание: MAE/WAPE оценивают прогноз, но не качество алертов без разметки инцидентов.")

## 5. Эксперимент: здесь начинаешь работать ТЫ

**Задание A.** Измени `min_drop_pct` сначала на 15%, затем на 40%. Что происходит с числом предупреждений? Почему «меньше тревог» не всегда лучше?

**Задание B.** Выбери одну строку с `alert=True`. Придумай **минимум три гипотезы** её появления. Как отличить: падение заказов, падение среднего чека, сбой выгрузки? Что нельзя подтвердить текущим набором данных?

**Задание C.** Измени baseline: используй 4-недельное **среднее** вместо медианы. На том же периоде проверки сравни WAPE и количество предупреждений. Какой метод выберешь и почему?

**Задание D (дополнительное).** Финансовый порог безубыточности: если фиксированные затраты за месяц — F, переменные затраты составляют долю v от выручки R, то в упрощённой модели прибыль `P = R*(1-v)-F`. Найди точку `P=0`. Почему даже это не помогает доказать реальный ущерб от отдельного аномального дня без дополнительных данных?

**После этого:** открой `src/model.py` и найди `add_features` и `backtest`. Сравни свою медиану с уже реализованным baseline, затем посмотри на ML и его оценку по времени. **Не подбирай пороги или модель по финальному тесту.**

**Что прислать для разбора:** три выбранных кандидата на аномалию, таблицу с MAE/WAPE для медианы и среднего, вывод по алертам, а также ответ о том, какая информация помогла бы подтвердить реальный инцидент.